# PHASE 10 - Ablation Study

This notebook measures the contribution of feature groups using the same CPU-only XGBoost configuration for every experiment. It does not use the test set to choose features.

A: time features only; B: time + weather; C: time + weather + lag; D: time + weather + lag + rolling features.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def find_project_root():
    for candidate in [Path("."), Path("..")]:
        if (candidate / "data/processed/train.csv").exists():
            return candidate
    raise FileNotFoundError("Run PHASE 3 first.")

project_root = find_project_root()
processed_dir = project_root / "data/processed"
metrics_dir = project_root / "results/metrics"
figures_dir = project_root / "results/figures"
train_df = pd.read_csv(processed_dir / "train.csv", parse_dates=["timestamp", "dteday"])
validation_df = pd.read_csv(processed_dir / "validation.csv", parse_dates=["timestamp", "dteday"])
test_df = pd.read_csv(processed_dir / "test.csv", parse_dates=["timestamp", "dteday"])

target_column = "cnt"
time_features = ["yr", "mnth", "hr", "holiday", "weekday", "workingday", "season", "hour", "day", "month", "year", "day_of_week", "day_of_year", "is_weekend", "is_workingday", "rush_hour"]
weather_features = ["weathersit", "temp", "atemp", "hum", "windspeed"]
lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]
rolling_features = ["rolling_mean_24", "rolling_mean_168"]
feature_sets = {
    "A_time_only": time_features,
    "B_time_weather": time_features + weather_features,
    "C_time_weather_lag": time_features + weather_features + lag_features,
    "D_time_weather_lag_rolling": time_features + weather_features + lag_features + rolling_features,
}

assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()
print(f"Project root: {project_root.resolve()}")
print(f"Experiments: {list(feature_sets)}")

## Fixed model configuration

These parameters were selected in PHASE 6 using validation RMSE. They stay fixed across all feature sets for a fair comparison.

In [ ]:
search_path = metrics_dir / "xgboost_validation_search.csv"
search_results = pd.read_csv(search_path).sort_values("RMSE")
best_row = search_results.iloc[0]
xgb_parameters = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "min_child_weight": int(best_row["min_child_weight"]),
}
print("Fixed XGBoost parameters:", xgb_parameters)

In [ ]:
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,
        "R2": r2_score(y_true, predictions),
    }

ablation_results = []
for experiment, features in feature_sets.items():
    model = XGBRegressor(
        **xgb_parameters,
        objective="reg:squarederror",
        tree_method="hist",
        device="cpu",
        n_jobs=-1,
        random_state=42,
    )
    start_time = perf_counter()
    model.fit(train_df[features], train_df[target_column])
    training_time = perf_counter() - start_time
    validation_predictions = model.predict(validation_df[features])
    test_predictions = model.predict(test_df[features])
    validation_metrics = calculate_metrics(validation_df[target_column], validation_predictions)
    test_metrics = calculate_metrics(test_df[target_column], test_predictions)
    ablation_results.extend([
        {"experiment": experiment, "feature_count": len(features), "split": "validation", **validation_metrics, "Training Time": training_time},
        {"experiment": experiment, "feature_count": len(features), "split": "test", **test_metrics, "Training Time": training_time},
    ])

ablation_results_df = pd.DataFrame(ablation_results)
display(ablation_results_df.round(4))
ablation_results_df.to_csv(metrics_dir / "ablation_study_metrics.csv", index=False)

In [ ]:
validation_results = ablation_results_df[ablation_results_df["split"] == "validation"].sort_values("RMSE")
test_results = ablation_results_df[ablation_results_df["split"] == "test"].sort_values("RMSE")
best_validation = validation_results.iloc[0]
best_test = test_results.iloc[0]
print(f"Best feature set by validation RMSE: {best_validation['experiment']} ({best_validation['RMSE']:.4f})")
print(f"Best feature set by test RMSE for reporting: {best_test['experiment']} ({best_test['RMSE']:.4f})")

plot_data = ablation_results_df.pivot(index="experiment", columns="split", values="RMSE")
ax = plot_data.plot(kind="bar", figsize=(12, 5))
ax.set_title("Ablation study: RMSE by feature set")
ax.set_ylabel("RMSE")
ax.set_xlabel("Feature set")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "ablation_study_rmse.png", dpi=150)
plt.show()

## Phase 10 conclusion

Compare validation rows to understand feature contribution without test leakage. The test rows provide the final held-out confirmation of the observed feature-engineering effect.